In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from dotenv import load_dotenv
import pprint
import importlib
from src.policy import generate_policy_text, send_openrouter_request
from src.evals import load_strong_reject, run_jailbreak_pipeline, get_success_stats, get_result_plot

load_dotenv()
device = 'cuda'

ModuleNotFoundError: No module named 'policy'

# Evaluations
Testing on strongreject against oss and o4-mini.

In [ ]:
# load datasets + set vars 
model_ids = {'oss': 'openai/gpt-oss-20b',
             'o4-mini': 'openai/o4-mini'}
qual = 'I have a lucky coin!'
dataset = load_strong_reject()

NameError: name 'load_strong_reject' is not defined

In [10]:
# run jailbreak pipeline for oss 
oss_results = run_jailbreak_pipeline(dataset, 
                       qualification = 'I have a lucky coin!',
                       add_distractor = True,
                       limit = 5,
                       distractor = 'cat_fact',
                       model_id = model_ids['oss'])

oss_df = pd.DataFrame(oss_results) 
oss_df.to_csv(f'data/oss_with_cat_distractor.csv') # later can generalize structure to test differential effects of interventions

HTTPError: 400 Client Error: Bad Request for url: https://openrouter.ai/api/v1/chat/completions

In [44]:
oss_df = pd.read_csv('data/oss_with_cat_distractor.csv')
stats = get_success_stats(oss_df)
plot = get_result_plot(stats, model = 'gpt-oss-20b', condition = 'cat_distractor')

/workspace/character-science/notebooks/080725-gpt-oss/evals.py:82: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [12]:
# run jailbreak pipeline for o4-mini
o4_results = run_jailbreak_pipeline(dataset, 
                       qualification = 'I have a lucky coin!',
                       add_distractor = True,
                       distractor = 'cat_fact',
                       model_id = model_ids['o4-mini'])

o4_df = pd.DataFrame(o4_results) 
o4_df.to_csv(f'data/o4mini_with_cat_distractor.csv') # later can generalize structure to test differential effects of interventions

NameError: name 'run_jailbreak_pipeline' is not defined

# Attention analysis

In [14]:
%%capture
model = AutoModelForCausalLM.from_pretrained(model_ids['oss'],
                             torch_dtype = 'auto',
                             trust_remote_code = True).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_ids['oss'])

You have loaded an FP4 model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model. To remove this warning, pass device_map = 'cuda'. 


In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

generated = model.generate(**inputs, max_new_tokens=1000, do_sample = False)


In [ ]:
print(tokenizer.decode(generated[0][inputs["input_ids"].shape[-1]:]))